In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
import pyreadr
from matplotlib.lines import Line2D
    
pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
### data load 

# load smoke datasets
epa_stations = gpd.read_file('../01_data/01_raw/childs_pm/epa_station_locations/epa_station_locations.shp')
station_smoke = pyreadr.read_r('../01_data/01_raw/childs_pm/station_smokePM_2025_01.rds')[None]

# load fires
fires = gpd.read_file('../01_data/01_raw/data_2025_01_17.geojson').to_crs(epsg=2229)
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.tz_convert('US/Pacific')
fires = fires[fires['poly_DateCurrent'] > '2025-01-06']
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.date
fires = fires[['geometry']]
fires_union = fires.dissolve()  # dissolve to one multipolygon
fires_union = fires_union.to_crs(epa_stations.crs)


# load and process counties
counties = gpd.read_file('~/Desktop/Desktop/epidemiology_PhD/01_data/clean/us_cnty_boundaries.geojson')
# define county groups
county_groups = {
    # group 1: LA County
    '06037': 'LA',
    
    # group 2: northwest counties
    '06111': 'Northwest',  # Ventura
    '06083': 'Northwest',  # Santa Barbara
    '06079': 'Northwest',  # San Luis Obispo
    '06029': 'Northwest',  # Kern
    
    # group 3: southeast counties
    '06059': 'Southeast',  # Orange
    '06073': 'Southeast',  # San Diego
    '06025': 'Southeast',  # Imperial
    '06065': 'Southeast',  # Riverside
    '06071': 'Southeast'   # San Bernardino
}
# list of counties to keep
county_fips_to_keep = list(county_groups.keys())
# filter counties dataset
counties_subset = counties[counties['fips'].isin(county_fips_to_keep)]
counties_subset = counties_subset.to_crs(epa_stations.crs)  # convert to same CRS as smoke

## EDA 

In [ ]:
epa_stations.explore()

In [ ]:
epa_stations.head()

# grid_5km: 5km grid they constructed, they assigned each monitor to each grid cell and then defined the smoke for each epa station that way.
# shouldn't matter here.
# these are the 5km grid cell IDs. 

In [ ]:
station_smoke
station_smoke[station_smoke['id'] == '060371103'].head(25)

# smoke_day: whether there was a smoke plume intersecting the 5km grid cell for that station on that day.
# LA_wildfire_day: is it plausible that that grid cell is affected by the LA wildfires. this var is trying to determine whether the smoke that day at that station was due to LA wildfires. maybe just use this variable as a stratifier to look at, but prob don't have to use it in the analysis. just nice to know who is exposed to what.
# pm25_med_3yr: median pm2.5 on non-smoke days amongst all days for that station in that month and the 2 years prior.
    # pm25 - pm25_med_3yr = pm25_anom because they're looking at the anomalous smoke above the 3 yr median. and then the smoekPM var indicates whether it was a smoke day.
    # when smoke_day is 1, when there is a pm25_anom, that is attributed to smokePM and thus is the value in smokePM.
    # smokePM is noisy but not necessarily wrong. just may include some other anomalous PM but is close.
# smokePM: what is the PM2.5 that we think is from wf smoke? 
# light/med/dense is a measure of smoke.

# probably use the smokePM variable bc thats the smoke pm. it will be 0 when it was not a smoke day.

# NOTE: there are no stations in the palisades fire area. since the wind blew toward the water, there was just nothing to pick up on smoke in that area bc all stations are behind it. not a ton of people affected by the palisades fire smoke, many more from eaton. we corroborated this with the modis satellite data and where there are missing data from the satellite, there just aren't a lot of people

In [ ]:
# combining the data 
station_smoke_gdf = epa_stations.merge(station_smoke, left_on='stn_id', right_on='id', how='left')
station_smoke_gdf = gpd.GeoDataFrame(station_smoke_gdf, crs=station_smoke_gdf.crs)

## Diagnostic plots 

1. Map: show both where the stations are as well as their PM values for each day the first week. Color them based on whether they are under a smoke plume and overlay the fires over on these plots. 
- one version with all of the region
- one version with just LA county
2. Time series showing each station's PM over the course of the first week
3. Map + time series by groups: Group the stations based on counties (defined below, generally Northwest, LA, Southeast) 

### First prep the data and subset to the appropriate geographic region and week!

In [ ]:
### prep data

# select columns and format date
st_smoke_gdf_sm = station_smoke_gdf[['stn_id', 'geometry', 'smokePM', 'smoke_day', 'date']]
st_smoke_gdf_sm['date'] = pd.to_datetime(st_smoke_gdf_sm['date'])

# subset to the right geo area and the right dates
# to get the right geographic area, subset based on the counties.
socal_data = gpd.sjoin(st_smoke_gdf_sm, counties_subset[['fips', 'geometry']], 
                                 how='inner', predicate='intersects')

# pull first week of LA fires programmatically
la_fires_start = pd.to_datetime('2025-01-07')
first_week = [
    la_fires_start,
    la_fires_start + pd.Timedelta(days=1),  # Jan 8
    la_fires_start + pd.Timedelta(days=2),  # Jan 9
    la_fires_start + pd.Timedelta(days=3),  # Jan 10
    la_fires_start + pd.Timedelta(days=4),  # Jan 11
    la_fires_start + pd.Timedelta(days=5),  # Jan 12
    la_fires_start + pd.Timedelta(days=6)   # Jan 13
]

### Plot 1: Map of stations and their smoke PM values each day of the first week
The map will also: 
1. indicate whether there is a smoke plume overhead
2. overlay the fires with the station data

In [ ]:
### plot 1 function
def plot_fire_smoke_pm(data, fires_gdf, first_week, title_suffix=""):
    """
    Plot smoke PM2.5 data with fire boundary overlays for a week of data.
    
    Parameters:
    -----------
    data : GeoDataFrame
        Air quality monitoring data with columns: date, smokePM, smoke_day, geometry
    fires_gdf : GeoDataFrame
        Fire boundary polygons to overlay
    first_week : list
        List of datetime objects for the week to plot
    title_suffix : str
        Additional text to add to plot title (e.g., "- LA County Only")
    """
    fig, axes = plt.subplots(7, 1, figsize=(10, 30))
    axes = axes.flatten()
    
    # Calculate color range from data
    q01 = data['smokePM'].quantile(0.01)
    q99 = data['smokePM'].quantile(0.99)
    vmin = max(0, q01) 
    vmax = q99
    
    print(f"Color scale range: {vmin:.2f} to {vmax:.2f} µg/m³")
    
    # Convert fires to Web Mercator for consistent plotting
    fires_mercator = fires_gdf.to_crs(epsg=3857)
    
    for i, day in enumerate(first_week):
        ax = axes[i]
        
        day_data = data[data['date'] == day]
        
        if len(day_data) == 0:
            ax.text(0.5, 0.5, 'No data\navailable', transform=ax.transAxes, 
                    ha='center', va='center', fontsize=12)
            ax.set_title(f'{day.strftime("%m/%d")}', fontsize=12, fontweight='bold')
            ax.axis('off')
            continue
        
        day_data_mercator = day_data.to_crs(epsg=3857)
        
        # Split data by smoke day status
        smoke_day_data = day_data_mercator[day_data_mercator['smoke_day'].notna()]
        no_smoke_day_data = day_data_mercator[day_data_mercator['smoke_day'].isna()]
        
        # Plot non-smoke day data as squares
        if len(no_smoke_day_data) > 0:
            no_smoke_day_data.plot(
                column='smokePM', 
                cmap='viridis', 
                markersize=80, 
                alpha=0.9,
                edgecolor='white',
                linewidth=1,
                ax=ax,
                vmin=vmin,
                vmax=vmax,
                marker='s'
            )
        
        # Plot smoke day data as circles
        if len(smoke_day_data) > 0:
            smoke_day_data.plot(
                column='smokePM', 
                cmap='viridis', 
                markersize=80, 
                alpha=0.9,
                edgecolor='white',
                linewidth=1,
                ax=ax,
                vmin=vmin,
                vmax=vmax,
                marker='o'
            )
        
        # Add fire boundaries
        fires_mercator.boundary.plot(ax=ax, color='red', linewidth=1, alpha=0.8)
        
        # Add basemap
        try:
            ctx.add_basemap(ax, 
                            crs=day_data_mercator.crs.to_string(), 
                            source=ctx.providers.OpenStreetMap.Mapnik,
                            alpha=0.6)
        except Exception as e:
            print(f"Basemap error for day {i+1}: {e}")
        
        ax.set_title(f'{day.strftime("%m/%d")}', fontsize=12)
        ax.axis('off')
        
        # Add statistics box
        smoke_count = len(smoke_day_data)
        no_smoke_count = len(no_smoke_day_data)
        day_stats = f'n={len(day_data)}\n□={no_smoke_count} ○={smoke_count}\nmax pm={day_data["smokePM"].max():.0f}'
        ax.text(0.02, 0.02, day_stats, transform=ax.transAxes, 
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8),
                verticalalignment='bottom', fontsize=8)
    
    # Add legends
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=8, label='No smoke day'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='Smoke day'),
        Line2D([0], [0], color='red', linewidth=2, label='Fire boundaries')
    ]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.95, 0.95))
    
    # Add colorbar
    fig.subplots_adjust(bottom=0.05, top=0.90, right=0.85, hspace=0.1)  # Made room on right side
    cbar_ax = fig.add_axes([0.87, 0.6, 0.03, 0.3])  # [left, bottom, width, height] - vertical on right
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')  # Changed to vertical
    cbar.set_label('Smoke PM₂.₅ (µg/m³)', fontsize=12)

    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n=== SUMMARY ===")
    for i, day in enumerate(first_week):
        day_data = data[data['date'] == day]
        print(f"Day {i+1} ({day.strftime('%Y-%m-%d %A')}): {len(day_data)} stations")
        if len(day_data) > 0:
            print(f"  PM range: {day_data['smokePM'].min():.1f}-{day_data['smokePM'].max():.1f} µg/m³")
            print(f"  Median PM: {day_data['smokePM'].median():.1f} µg/m³")
            high_pm_count = (day_data['smokePM'] > 10).sum()
            print(f"  Stations with PM > 10: {high_pm_count} ({high_pm_count/len(day_data)*100:.1f}%)")
        else:
            print("  No data available")
        print()


In [ ]:
### plot 1 (all counties)
plot_fire_smoke_pm(socal_data, fires_union, first_week)

In [ ]:
### plot 1 (LA only)
la_county = counties_subset[counties_subset['fips'] == '06037']
la_county = la_county.drop(columns=['index_right'], errors='ignore')
socal_data = socal_data.drop(columns=['index_right'], errors='ignore')
la_county_data = gpd.sjoin(socal_data, la_county, 
                           how='inner', predicate='intersects')
plot_fire_smoke_pm(la_county_data, fires_union, first_week, " - LA County Only")

## Plot 2: Time series showing each station's PM over the course of the first week

In [ ]:
### plot 2: make a time series line plot of station pm values

# filter to first week and remove NAs
first_week_data = socal_data[socal_data['date'].isin(first_week)].copy()
first_week_data = first_week_data.dropna(subset=['smokePM'])

plt.figure(figsize=(12, 8))
unique_stations = first_week_data['stn_id'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_stations)))

# plot each station as a separate line using colors from the colormap
for i, station in enumerate(unique_stations):
    station_data = first_week_data[first_week_data['stn_id'] == station]
    
    # sort by date to ensure proper line connections
    station_data = station_data.sort_values('date')
    
    plt.plot(station_data['date'], station_data['smokePM'], 
             marker='o', linewidth=2, markersize=6,
             label=f'Station {station}', color=colors[i])

plt.title('PM2.5 levels by station - January 7-13, 2025', 
          fontsize=16, pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('PM2.5 (μg/m³)', fontsize=12)

plt.xticks(first_week, [date.strftime('%b %d') for date in first_week], rotation=45)

plt.tight_layout()
plt.ylim(bottom=0)
plt.show()


## Plot 3: Map + time series by groups: Group the stations based on counties (defined below, generally Northwest, LA, Southeast) 

In [ ]:
### plot 3: prep for plotting map + time series: create groups based on location 

# group 1: LA county (06037)
# group 2: ventura (06111), santa barbara (06083), san louis obispo (06079), kern (06029)
# group 3: orange (06059), san diego (06073), imperial (06025), riverside (06065), san bernadino (06071)

# spatial join to get FIPS codes in station data
counties_subset = counties_subset.to_crs(socal_data.crs)
plot_df = socal_data.copy()

# assign groups based on FIPS codes
plot_df['group'] = plot_df['fips'].map(county_groups)
plot_df = plot_df.drop(columns=['index_right'], errors='ignore')
plot_df

In [ ]:
def create_combined_plot(plot_df, first_week, fires_union=None, save_path=None):
    """
    Side by side plot: map on left, time series on right
    Now includes fire boundaries on the map
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    plt.subplots_adjust(hspace=0, wspace=0.1)
    
    # color mapping for regions
    region_colors = {
        'Northwest': '#66C2A5',
        'LA': '#FC8D62', 
        'Southeast': '#8DA0CB'
    }
    
    # define region order for consistent legend
    region_order = ['Northwest', 'LA', 'Southeast']
    
    # LEFT PLOT: map
    # first add fire boundaries first (so they appear behind the points)
    if fires_union is not None:
        fires_union.plot(ax=ax1,
                        facecolor='red',
                        edgecolor='darkred',
                        alpha=0.3,
                        linewidth=1.5)
    # then add the station points
    for group in region_order:
        if group in plot_df['group'].values:
            group_df = plot_df[plot_df['group'] == group]
            group_df.plot(ax=ax1,
                         color=region_colors[group],
                         markersize=125,
                         alpha=1,
                         edgecolor=region_colors[group],
                         linewidth=0.5)
    # finally add basemap
    ctx.add_basemap(ax1,
                   crs=plot_df.crs,
                   source=ctx.providers.CartoDB.Positron,
                   zoom=10)
    
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    
    
    # RIGHT PLOT: time series (unchanged)
    # filter to first week and remove NA values
    first_week_data = plot_df[
        plot_df['date'].isin(first_week)
    ].dropna(subset=['smokePM'])
    
    # plot each station grouped by region in specified order
    for group in region_order:
        if group in first_week_data['group'].values:
            group_df = first_week_data[first_week_data['group'] == group]
            unique_stations = group_df['stn_id'].unique()
            
            for i, station in enumerate(unique_stations):
                station_data = group_df[group_df['stn_id'] == station].sort_values('date')
                
                ax2.plot(station_data['date'], station_data['smokePM'],
                        color=region_colors[group],
                        marker='o',
                        linewidth=2,
                        markersize=4,
                        alpha=0.8,
                        label=group if i == 0 else "")  # label once per region
    
    fig.suptitle('PM$_{2.5}$ by station and group - January 7-13, 2025',
                 fontsize=20, y=0.98)
    ax2.set_ylabel('PM$_{2.5}$ (μg/m³)', fontsize=14)
    ax2.set_xticks(first_week)
    ax2.set_xticklabels([date.strftime('%m/%d') for date in first_week], fontsize=12)
    ax2.tick_params(axis='y', labelsize=12)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    
    ax2.set_ylim(bottom=0)
    
    ax2.legend(title='Group', fontsize=14, title_fontsize=14)
    
    plt.tight_layout()
    
    # save
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")
    
    plt.show()


In [ ]:
### plot 3 without fires
create_combined_plot(plot_df, first_week, save_path='../03_output/station_map_time_series.png')

In [ ]:
### plot 3 with fire boundaries
create_combined_plot(plot_df, first_week, fires_union=fires_union, save_path='../03_output/station_map_time_series_with_fires.png')

In [ ]:
def plot_weekly_average_pm(data, fires_gdf, first_week):
        
    # filter data for the week period
    week_data = data[data['date'].isin(first_week)]
    
    if len(week_data) == 0:
        print("No data available for the specified week period")
        return
    
    # calc average PM2.5 for each station over the week
    # create a station identifier using coordinates
    
    station_averages = week_data.groupby('stn_id').agg({
        'smokePM': 'mean',
        'geometry': 'first'  # Keep the geometry
    }).reset_index()
    
    # convert back to GeoDataFrame and preserve CRS
    station_averages = gpd.GeoDataFrame(station_averages, geometry='geometry', crs=week_data.crs)
    
    # calc color range (same as original)
    q01 = data['smokePM'].quantile(0.01)
    q99 = data['smokePM'].quantile(0.99)
    vmin = max(0, q01) 
    vmax = q99
    
    print(f"Color scale range: {vmin:.2f} to {vmax:.2f} µg/m³")
    print(f"Average PM range: {station_averages['smokePM'].min():.2f} to {station_averages['smokePM'].max():.2f} µg/m³")
    
    # make the plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # convert to Web Mercator for consistent plotting
    station_averages_mercator = station_averages.to_crs(epsg=3857)
    fires_mercator = fires_gdf.to_crs(epsg=3857)
    
    # plot stations colored by average PM2.5
    station_averages_mercator.plot(
        column='smokePM', 
        cmap='viridis', 
        markersize=175, 
        alpha=0.9,
        edgecolor='none',
        linewidth=1.5,
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        marker='o'
    )
    
    # add fire boundaries
    fires_mercator.boundary.plot(ax=ax, color='red', linewidth=2, alpha=0.8)
    
    # add basemap
    try:
        ctx.add_basemap(ax, 
                        crs=station_averages_mercator.crs.to_string(), 
                        source=ctx.providers.CartoDB.Positron,
                        alpha=0.6)
    except Exception as e:
        print(f"Basemap error: {e}")
    
    week_start = first_week[0].strftime("%m/%d")
    week_end = first_week[-1].strftime("%m/%d")
    ax.set_title(f'Weekly average smoke PM₂.₅ ({week_start} - {week_end})', 
                 fontsize=14, pad=20)
    ax.axis('off')
    
    # add stats
    n_stations = len(station_averages)
    avg_pm = station_averages['smokePM'].mean()
    max_pm = station_averages['smokePM'].max()
    high_pm_count = (station_averages['smokePM'] > 10).sum()
    
    stats_text = f'Stations: {n_stations}\nWeek avg: {avg_pm:.1f} µg/m³\nMax avg: {max_pm:.1f} µg/m³\nAvg > 10 µg/m³: {high_pm_count} ({high_pm_count/n_stations*100:.1f}%)'
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9),
            verticalalignment='top', fontsize=10)
    
    # add legend for fire boundaries
    legend_elements = [
        Line2D([0], [0], color='red', linewidth=2, label='Fire boundaries')
    ]
    ax.legend(handles=legend_elements, loc='lower left')
    
    # add colorbar
    cbar_ax = fig.add_axes([0.03, 0.15, 0.3, 0.03])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Average Smoke PM₂.₅ (µg/m³)', fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_weekly_average_pm(socal_data, fires_union, first_week)